In [64]:
import bisect
class Index(object):            #Create an index object
    def __init__(self, t, k):   #Function for initializing the lists with all the substrings of kmer with length k and their offsets in the text t
        self.k = k
        self.index = []
        for i in range(len(t) -k + 1):      #Itertate through text t to find all possible kmers of length k
            self.index.append((t[i:i + k], i))  #Append the index list with tuple containing kmer substring and their position
        self.index.sort()                       #Sort the index list

    def query(self, p):
        """Function for querying the index list for pattern p"""  
                   
        kmer = p[:self.k]           # Extract first kmer of length k from pattern p 
        i = bisect.bisect_left(self.index, (kmer, -1)) #Finds the 1st position in the list where kmer occurs, -1 position ensures the first occurence of the kmer
        hits = []
        while i < len(self.index):          #Iteration starts at i obtained from bisect and continues along the index list
            if self.index[i][0] != kmer:        #Checks the first elemnt of the tuple i.e, "kmer-string" at position i (resulted from bisect) in a an index
                break                           # Breaks the loop if kmer not matched as the index list is sorted
            hits.append(self.index[i][1])       #Append the second elemnt of the tuple i.e, offset of that kmer at position i (resulted from bisect) in an index list
            i += 1
        return hits   #list containing all the indices in text t where kmer extracted from p matches
         


In [3]:
def queryindex(p, t, index):
    k = index.k             #set the length of the kmer from an index object
    offsets = []
    for i in index.query(p):        #iterates over all the hits indices obtained from the query function
        if p[k:] == t[i + k: i + len(p)]:  #verify the rest of the patttern in the text
            offsets.append(i)
    return offsets                      #return all the indices where pattern matches exactly


In [65]:
import bisect
   
class SubseqIndex(object):
    """ Holds a subsequence index for a text T """
    
    def __init__(self, t, k, ival):
        """ Create index from all subsequences consisting of k characters
            spaced ival positions apart.  E.g., SubseqIndex("ATAT", 2, 2)
            extracts ("AA", 0) and ("TT", 1). """
        self.k = k  # num characters per subsequence extracted
        self.ival = ival  # space between them; 1=adjacent, 2=every other, etc
        self.subseqindex = []
        self.span = 1 + ival * (k - 1)
        for i in range(len(t) - self.span + 1):  # for each subseq
            self.subseqindex.append((t[i:i+self.span:ival], i))  # add (subseq, offset)
        self.subseqindex.sort()  # alphabetize by subseq
    
    def subseq_query(self, p):
        """ Return index hits for first subseq of p """
        subseq = p[:self.span:self.ival]  # query with first subseq
        i = bisect.bisect_left(self.subseqindex, (subseq, -1))  # binary search
        hits = []
        while i < len(self.subseqindex):  # collect matching index entries
            if self.subseqindex[i][0] != subseq:
                break
            hits.append(self.subseqindex[i][1])
            i += 1
        return hits
    
def querysubseqindex(p, t, subseqindex):
    k = subseqindex.k             #set the length of the kmer from an index object
    offsets = []
    
    for i in subseqindex.subseq_query(p):        #iterates over all the hits indices obtained from the query function
        if p[k:] == t[i + k: i + len(p)]:  #verify the rest of the patttern in the text
            offsets.append(i)
            
    
    return offsets                      #return all the indices where pattern matches exactly
    

In [ ]:
def readGenome(filename):
    genome = ""
    with open(filename,'r') as f:
        for line in f:
            if not line[0] == ">":
                genome += line.rstrip()
    return genome


In [66]:
"""Example below:"""
pattern = 'GGCGCGGTGGCTCACGCCTGTAAT'

text = readGenome(r"F:\Coursera\Algorithms for DNA sequences\Module_2\chr1.GRCh38.excerpt.fasta")

index = Index(text, 8)      #Kmer can be of any length less than length of pattern

subseq_index = SubseqIndex(text, 8, 3)
my_text = 'to-morrow and to-morrow and to-morrow creeps in this petty pace'
my_pattern = "to-morrow and to-morrow "
my_subseqindex = SubseqIndex(my_text, 8, 3)
allowed_edits = 2


In [69]:
def approximate_match(p,t,n):
    segment_length = int(round(len(p) / (n+1)))
    all_matches = set()     #initialized the variable as set to prevent the duplication of the same location of pattern in text
    index_hits = 0
    for i in range(n+1):   #iterate over each segments of p i.e., n+1
        start = i * segment_length
        end = min((i + 1) * segment_length, len(p))  #minimum from next segment length and pattern length which actualize the length of the last segment to prevent the positions to run past the pattern
        matches = queryindex(p[start:end], t, index)    #return lists of positions where segment of pattern matches the text
        index_hits += len(matches) #count the number of index hits
        for m in matches:
            if m < start or m-start + len(p) > len(t): #prevents pattern from running off the begining (m < start) or end (m-start+ len(p)>len(t)) of the text
                continue        #skip the rest of the loop and continue with next position of matches
            
            mismatches = 0
            for j in range(0, start):   #check for the segment before the start of a segment
                if not p[j] == t[m-start+j]:     #compares each character of pattern(from begining of pattern to segment start) with character of text from possible start of matching location up to match location
                    mismatches += 1
                    if mismatches > n:      #if mismatches exceed the allowed edits then break the loop
                        break
            
            for j in range(end, len(p)):    #check for the segment after the end of the segment
                if not p[j] == t[m-start+j]:    #compares each character of pattern(segment end to end of the pattern) with character of text from macth location up to possible end of matching location
                    mismatches += 1
                    if mismatches > n:      
                        break

            if mismatches <= n:
                all_matches.add(m - start)  # (m - start) corresponds to the begining of the pattern
    print(index_hits)
    return list(all_matches)        #convert the set into list
    

In [ ]:
result_1 = approximate_match(pattern, text, allowed_edits)
print(len(result_1))


90
19


In [ ]:
"""Code blocks from this cell below is for validating the result of approximate matching with naive exact matching for the same pattern and text"""


In [ ]:

def naive(p,t):
    """Matches the occurences of pattern 'p' in text 't'."""
    occurences = []
    for i in range(len(t)-len(p)+1):
        match = True
        for j in range(len(p)):
            if not t[i+j] == p[j]:
                match = False
                break
        if match:
            occurences.append(i)
    return occurences


In [ ]:
def indexhitsWithnaive(p,t,n):
    segment_length = int(round(len(p) / (n+1)))    
    index_hits = 0
    for i in range(n+1):   #iterate over each segments of p i.e., n+1
        start = i * segment_length
        end = min((i + 1) * segment_length, len(p))  #minimum from next segment length and pattern length which actualize the length of the last segment to prevent the positions to run past the pattern
        matches = naive(p[start:end], t)    #return lists of positions where segment of pattern matches the text
        index_hits += len(matches) #count the number of index hits
    return index_hits
print(indexhitsWithnaive(pattern, text, allowed_edits))


90


In [ ]:
def naive_2mm(p,t, distance):
    """Matches the occurences of pattern 'p' in text 't' with defined mismatches."""
    occurences = []
    for i in range(len(t)-len(p)+1):
        match = True
        mismatch = 0
        for j in range(len(p)):
            if t[i+j] == p[j]:
                match
            if not t[i+j] == p[j]:
                mismatch += 1
            if mismatch > distance: #Defined mismatches to '2'. It is customizable.
                match = False
                break
        if match:
            occurences.append(i)
    return occurences
total = naive_2mm(pattern, text, allowed_edits) #Example of counting the number of occurences of a pattern in a genome
print(len(total))


19
